#### NOTES DE COHERENCE (**à retirer à postériori**)
- Analyse si il y a pas de logement observé et/ou pas de carte d'électeurs vs type de ménage
- Prendre en considération si c'est le/la cheffe de ménage qui répond ou non et son statut mat / decision / status
- Prendre en considération le lieu (village)
- Vérifier que l'âge a été "constraint" - 🟠 Vérification de <span style="color: orange">niveau intermédiaire</span> : ⚠️ voir dans quels cas les mineurs sont autorisés à participer à l'enquête (possibilité d'ajout de questions représentants/accompagnées) -> *vérification technique* 
- Vérifier les questions de protection - 🔴 Vérification de <span style="color: red">niveau élevé</span> -> 

## 1. Analyse Univariée de la Table principale

In [1]:
import sqlite3
import re
import pandas as pd

# ==========================================
# 1. CONNEXION ET CHARGEMENT DES DONNÉES
# ==========================================
db_path = r"C:\Users\maril\Documents\Enquête Multisectorielle - ONG\outputs\enquete_ms.db"
conn = sqlite3.connect(db_path)

nom_table_reelle = "META_ENQUETE"
df_data = pd.read_sql(f"SELECT * FROM {nom_table_reelle}", conn)
conn.close()

chemin_xlsform = r"C:\Users\maril\Documents\Enquête Multisectorielle - ONG\data\Questionnaire Vulnérabilités Fictif.xlsx"
df_survey = pd.read_excel(chemin_xlsform, sheet_name="survey")

print(f"✅ Données chargées : {len(df_data)} lignes et {len(df_data.columns)} colonnes.")

# --- LISTES D'EXCLUSION ÉLARGIES ---
COLONNES_EXCLUES_TECHNIQUES = ['id', 'uuid', 'index', 'jeton', 'token', 'key', '_id', '_uuid']
MOTS_CLES_EXCLUS_TEMPORELS = ['start', 'end', 'today', 'deviceid', 'subscriberid', 'simserial', 'phonenumber', 'date', 'heure']


# ==========================================
# 2. PROPAGATION ROBUSTE DES GROUPES ET TRADUCTION
# ==========================================
def preparer_regles_survey(df_survey):
    lignes_traitees = []
    pile_groupes = []
    
    for _, row in df_survey.iterrows():
        col_type = str(row.get('type', ''))
        col_name = row.get('name')
        relevant_actuel = row.get('relevant')
        
        if col_type.startswith('begin_group') or col_type.startswith('begin group'):
            pile_groupes.append(relevant_actuel)
            continue
        elif col_type.startswith('end_group') or col_type.startswith('end group'):
            if pile_groupes:
                pile_groupes.pop()
            continue
            
        if pd.notna(col_name):
            tous_les_relevant = []
            for g_rel in pile_groupes:
                if pd.notna(g_rel) and str(g_rel).strip():
                    tous_les_relevant.append(str(g_rel).strip())
            
            if pd.notna(relevant_actuel) and str(relevant_actuel).strip():
                tous_les_relevant.append(str(relevant_actuel).strip())
                
            combined_relevant = " & ".join([f"({r})" for r in tous_les_relevant]) if tous_les_relevant else None
            
            row_dict = row.to_dict()
            row_dict['relevant_combine'] = combined_relevant
            lignes_traitees.append(row_dict)
            
    return pd.DataFrame(lignes_traitees)


def traduire_relevant_en_pandas(relevant_expr):
    if pd.isna(relevant_expr) or not str(relevant_expr).strip():
        return None
        
    expr_pandas = str(relevant_expr)
    
    if "selected(" in expr_pandas:
        match = re.search(r"selected\(\s*\${([^}]+)}\s*,\s*'([^']+)'\)", expr_pandas)
        if match:
            col, val = match.groups()
            expr_pandas = f"df_data['{col}'] == '{val}'"
    
    expr_pandas = re.sub(r'\$\{([^}]+)\}', r"df_data['\1']", expr_pandas)
    expr_pandas = expr_pandas.replace(' and ', ' & ').replace(' or ', ' | ')
    
    return expr_pandas


def rendre_relevant_lisible(expr_brute):
    """Transforme une expression XLSForm brute en phrase claire et compréhensible."""
    if pd.isna(expr_brute) or not str(expr_brute).strip():
        return ""
    
    texte = str(expr_brute).strip()
    
    # Nettoyage récursif des parenthèses englobantes
    while texte.startswith('(') and texte.endswith(')'):
        texte = texte[1:-1].strip()
        
    texte = re.sub(r'\$\{([^}]+)\}', r'\1', texte)
    texte = texte.replace('selected(', '').replace(')', '')
    
    if ',' in texte:
        parts = [p.strip().replace("'", "").replace('"', '') for p in texte.split(',')]
        if len(parts) == 2:
            return f"La question s'applique si la variable '{parts[0]}' vaut '{parts[1]}'"
    elif '==' in texte:
        parts = [p.strip().replace("'", "").replace('"', '') for p in texte.split('==')]
        if len(parts) == 2:
            return f"La question s'applique si la variable '{parts[0]}' vaut '{parts[1]}'"
            
    return f"Condition : {texte}"


# ==========================================
# 3. MOTEUR D'ANALYSE UNIVARIÉE INTELLIGENT
# ==========================================
def executer_eda_univariee(df_data, df_survey_clean):
    total_global = len(df_data)
    colonnes_xlsform = set(df_survey_clean['name'].dropna().tolist())
    colonnes_orphelines = list(set(df_data.columns) - colonnes_xlsform)
    
    for _, row in df_survey_clean.iterrows():
        col_name = row.get('name')
        col_type = str(row.get('type'))
        relevant_rule = row.get('relevant_combine')
        
        if pd.isna(col_name) or col_name not in df_data.columns:
            continue
            
        if any(kw in col_name.lower() for kw in COLONNES_EXCLUES_TECHNIQUES + MOTS_CLES_EXCLUS_TEMPORELS):
            continue
            
        df_cible = df_data.copy()
        filtre_pandas = traduire_relevant_en_pandas(relevant_rule)
        
        total_eligible = total_global
        masque_valide = True
        
        if filtre_pandas:
            try:
                mask = eval(filtre_pandas)
                df_cible = df_data[mask]
                total_eligible = len(df_cible)
            except Exception as e:
                masque_valide = False
                total_eligible = 0 

        print(f"\n==================================================")
        print(f"🔍 Analyse : {col_name} (Type XLSForm : {col_type})")
        print(f"--------------------------------------------------")
        
        if filtre_pandas and masque_valide:
            print(f"📌 Conditionnement : {rendre_relevant_lisible(relevant_rule)}")
            print(f"   - Éligibles : {total_eligible} / {total_global} lignes")
        elif filtre_pandas and not masque_valide:
            print(f"📌 Conditionnement : {relevant_rule}")
            print(f"   - ⚠️ Erreur d'évaluation du filtre - Vérifiez la syntaxe")
            
        if total_eligible == 0 and filtre_pandas:
            print("⚠️ Aucun enregistrement ou échec du filtre éligible.")
            continue
            
        # --- INFÉRENCE INTELLIGENTE DES TYPES ---
        if col_type == 'text':
            series_val = df_cible[col_name].dropna()
            nb_uniques = series_val.nunique()
            if 0 < nb_uniques <= 6:
                print(f"💡 [Inférence] Type 'text' converti en select_one (faible cardinalité : {nb_uniques} valeurs uniques).")
                col_type = 'select_one_inferred'

        # --- 1. STATISTIQUES NUMÉRIQUES ---
        if col_type.startswith('integer') or col_type.startswith('decimal'):
            nb_nuls = df_cible[col_name].isnull().sum()
            print(f"📊 Statistiques Numériques (Valeurs manquantes : {nb_nuls}) :")
            stats = df_cible[col_name].describe()[['mean', 'std', 'min', '50%', 'max']]
            stats.index = ['Moyenne', 'Écart-type', 'Minimum', 'Médiane', 'Maximum']
            print(stats.round(2))
            
        # --- 2. TABLEAUX DE FRÉQUENCES (CATÉGORIELS) ---
        elif col_type.startswith('select_one') or col_type.startswith('select_multiple') or col_type == 'select_one_inferred':
            freq = df_cible[col_name].value_counts(dropna=False)
            freq_pct = (freq / total_eligible) * 100
            
            resume = pd.DataFrame({
                'Effectif': freq, 
                'Pourcentage (%)': freq_pct.round(2)
            })
            
            resume.index = ['NaN (non concernés)' if pd.isna(x) else str(x) for x in resume.index]
            
            total_effectif = resume['Effectif'].sum()
            total_pct = (total_effectif / total_eligible) * 100 if total_eligible > 0 else 0
            df_total = pd.DataFrame({
                'Effectif': [total_effectif],
                'Pourcentage (%)': [round(total_pct, 2)]
            }, index=['TOTAL'])
            
            print(f"📋 Tableau de fréquences (Total catégories : {len(freq)}) :")
            if len(freq) > 20:
                print(pd.concat([resume.head(25), df_total]).to_string())
            else:
                print(pd.concat([resume, df_total]).to_string())

        # --- 3. NLP POUR LES TEXTES LIBRES LONGS ---
        elif col_type == 'text':
            series_text = df_cible[col_name].dropna().astype(str)
            series_text = series_text[series_text.str.strip() != '']
            print(f"📝 Analyse NLP / Texte libre ({len(series_text)} réponses) :")
            
            if len(series_text) > 0:
                longueurs_mots = series_text.apply(lambda x: len(x.split()))
                longueurs_car = series_text.apply(len)
                print(f"   - Longueur moyenne : {longueurs_mots.mean():.1f} mots ({longueurs_car.mean():.1f} caractères)")
                
                stopwords_fr = {'le', 'la', 'les', 'un', 'une', 'des', 'et', 'en', 'à', 'de', 'du', 'au', 'aux', 'que', 'qui', 'dans', 'sur', 'par', 'pour', 'avec', 'pas', 'plus', 'sont', 'est', 'ont', 'nous', 'vous', 'ils', 'elles', 'leur', 'leurs', 'son', 'sa', 'ses', 'ce', 'cette', 'ces', 'mais', 'ou', 'donc', 'or', 'ni', 'car', 'etre', 'avoir', 'suis', 'es', 'sommes', 'etes', 'sont', 'ai', 'as', 'avons', 'avez', 'ont', 'tout', 'tous', 'toute', 'toutes', 'faire', 'fait'}
                
                tous_les_mots = " ".join(series_text).lower()
                mots_propres = re.findall(r'\b[a-zA-Zàâäéèêëïîôöùûüç]{4,}\b', tous_les_mots)
                mots_filtres = [m for m in mots_propres if m not in stopwords_fr]
                
                top_mots = pd.Series(mots_filtres).value_counts().head(5)
                print(f"   - Top mots significatifs : {dict(top_mots)}")
                
                print(f"   - Top 10 des verbatims / commentaires les plus fréquents (extraits) :")
                top_commentaires = series_text.value_counts().head(10)
                for comm, count in top_commentaires.items():
                    print(f"     * ({count}x) \"{comm}\'")
            else:
                print("   - Aucune réponse textuelle enregistrée.")

    # --- 4. TRAITEMENT AUTOMATIQUE DES COLONNES DE NETTOYAGE (ex: duree_minutes) ---
    print(f"\n==================================================")
    print(f"🔍 ANALYSE DES COLONNES CALCULÉES / NETTOYAGE (HORS XLSFORM)")
    print(f"==================================================")
    
    for col_name in colonnes_orphelines:
        if any(kw in col_name.lower() for kw in COLONNES_EXCLUES_TECHNIQUES):
            continue
            
        if pd.api.types.is_numeric_dtype(df_data[col_name]):
            print(f"\n📊 Statistiques Numériques (Calculée) : {col_name}")
            nb_nuls = df_data[col_name].isnull().sum()
            print(f"   - Valeurs manquantes : {nb_nuls}")
            stats = df_data[col_name].describe()[['mean', 'std', 'min', '50%', 'max']]
            stats.index = ['Moyenne', 'Écart-type', 'Minimum', 'Médiane', 'Maximum']
            print(stats.round(2))
        else:
            print(f"\n📋 Analyse Catégorielle (Calculée) : {col_name}")
            freq = df_data[col_name].value_counts(dropna=False)
            freq_pct = (freq / total_global) * 100
            resume = pd.DataFrame({'Effectif': freq, 'Pourcentage (%)': freq_pct.round(2)})
            resume.index = ['NaN (non concernés)' if pd.isna(x) else str(x) for x in resume.index]
            print(resume.head(10).to_string())

# ==========================================
# 4. LANCEMENT DU MOTEUR
# ==========================================
df_survey_clean = preparer_regles_survey(df_survey)
executer_eda_univariee(df_data, df_survey_clean)

✅ Données chargées : 50000 lignes et 13 colonnes.

🔍 Analyse : enqueteur (Type XLSForm : select_one enqueteur)
--------------------------------------------------
📋 Tableau de fréquences (Total catégories : 25) :
              Effectif  Pourcentage (%)
enqueteur_2       2088             4.18
enqueteur_12      2070             4.14
enqueteur_5       2053             4.11
enqueteur_10      2049             4.10
enqueteur_22      2044             4.09
enqueteur_7       2039             4.08
enqueteur_16      2038             4.08
enqueteur_11      2029             4.06
enqueteur_6       2018             4.04
enqueteur_19      2010             4.02
enqueteur_21      2010             4.02
enqueteur_4       2007             4.01
enqueteur_18      2006             4.01
enqueteur_14      1994             3.99
enqueteur_17      1993             3.99
enqueteur_15      1990             3.98
enqueteur_9       1987             3.97
enqueteur_3       1973             3.95
enqueteur_13      1970      

## 2. Analyse univariée des tables secondaires

In [4]:
import json
import sqlite3
import re
import unicodedata
from pathlib import Path
import pandas as pd

# ==========================================
# 1. LISTE DES TABLES SECONDAIRES D'APRES LE MPD
# ==========================================
tables_secondaires = [
    "STATUT_MENAGE",
    "COMPOSITION_MENAGE",
    "VULNERABILITE_SANTE",
    "ECONOMIE_REVENU",
    "SECURITE_ALIMENTAIRE",
    "BIENS_AME",
    "WASH_EAU",
    "AGRICULTURE",
    "PROTECTION"
]

COLONNES_EXCLUES_TECHNIQUES = ['id', 'uuid', 'index', 'jeton', 'token', 'key', '_id', '_uuid', 'id_menage', 'parent_index', '_parent_index']
MOTS_CLES_EXCLUS_TEMPORELS = ['start', 'end', 'today', 'deviceid', 'subscriberid', 'simserial', 'phonenumber', 'date', 'heure', 'timestamp']
recommandations_conditions = None
for base in [Path.cwd(), *Path.cwd().parents]:
    chemin_edna = base / 'outputs' / 'edna_mode_capit.json'
    if chemin_edna.is_file():
        with chemin_edna.open(encoding='utf-8') as fichier_edna:
            historique_edna = json.load(fichier_edna)
        rapports_edna = historique_edna if isinstance(historique_edna, list) else [historique_edna]
        for rapport in reversed(rapports_edna):
            recommandations = rapport.get('capitalisation_edna_mode', {}).get('recommandations_conditions_analyse') if isinstance(rapport, dict) else None
            if recommandations:
                recommandations_conditions = recommandations
                break
        if recommandations_conditions:
            break
if recommandations_conditions is None:
    raise KeyError('Aucune recommandation d\u2019analyse n\u2019est enregistr\u00e9e dans outputs/edna_mode_capit.json.')


def afficher_alerte_coherence(message):
    print(f'\n\033[91m\U0001f6a8 Alerte de coh\u00e9rence : {message}\033[0m\n')


def normaliser_reponse(valeur):
    texte = unicodedata.normalize('NFKD', str(valeur).replace(chr(8217), ' ').replace(chr(39), ' ')).encode('ascii', 'ignore').decode('ascii').lower()
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9_]+', ' ', texte)).strip()


def preparer_choix(df_choices):
    choix_par_liste = {}
    for _, choix in df_choices.dropna(subset=['list_name', 'name']).iterrows():
        nom_liste = str(choix['list_name']).strip()
        nom = str(choix['name']).strip()
        label = str(choix.get('label', nom)).strip()
        if not nom_liste or not nom:
            continue
        aliases = {normaliser_reponse(nom), normaliser_reponse(label)}
        aliases.discard('')
        choix_par_liste.setdefault(nom_liste, []).append({
            'name': nom,
            'label': label if label and label.lower() != 'nan' else nom,
            'aliases': aliases
        })
    for choix_liste in choix_par_liste.values():
        choix_liste.sort(key=lambda choix: max(map(len, choix['aliases'])), reverse=True)
        motifs = [(re.compile(r'(?<!\w)' + re.escape(alias) + r'(?!\w)'), choix['name'])
                  for choix in choix_liste for alias in choix['aliases']]
        motifs.sort(key=lambda element: len(element[0].pattern), reverse=True)
        choix_liste.append({'_motifs': motifs})
    return choix_par_liste


def extraire_choix_multiples(reponse, choix_liste):
    texte = normaliser_reponse(reponse)
    if not texte or texte == 'nan':
        return set()
    selectionnes = set()
    positions_occupees = []
    motifs = choix_liste[-1].get('_motifs', [])
    for motif, nom in motifs:
        for match in motif.finditer(texte):
            if any(match.start() < fin and match.end() > debut for debut, fin in positions_occupees):
                continue
            selectionnes.add(nom)
            positions_occupees.append((match.start(), match.end()))
    return selectionnes


def inferer_type_texte(series, choix_par_liste):
    reponses = series.dropna().astype(str).str.strip()
    reponses = reponses[reponses.ne('')].map(normaliser_reponse).value_counts()
    if reponses.empty:
        return None, None

    meilleurs = []
    for nom_liste, choix_liste in choix_par_liste.items():
        alias_vers_choix = {alias: choix['name'] for choix in choix_liste[:-1] for alias in choix['aliases']}
        couverture_simple = sum(n for reponse, n in reponses.items() if reponse in alias_vers_choix) / reponses.sum()
        if couverture_simple >= 0.8:
            meilleurs.append((couverture_simple, 'select_one_inferred', nom_liste))
            continue
        if len(reponses) > 500:
            continue
        parsees = [(reponse, n, extraire_choix_multiples(reponse, choix_liste)) for reponse, n in reponses.items()]
        couverture_multi = sum(n for _, n, selections in parsees if selections) / reponses.sum()
        part_multi = sum(n for _, n, selections in parsees if len(selections) >= 2) / reponses.sum()
        if couverture_multi >= 0.8 and part_multi >= 0.2:
            meilleurs.append((couverture_multi + part_multi, 'select_multiple_inferred', nom_liste))
    if not meilleurs:
        return None, None
    _, type_inferre, nom_liste = max(meilleurs)
    return type_inferre, nom_liste


def evaluer_relevant(expression, dataframe, df_survey_clean, choix_par_liste):
    texte = str(expression)
    colonnes_survey = {
        str(row['name']): str(row.get('type', ''))
        for _, row in df_survey_clean.iterrows() if pd.notna(row.get('name'))
    }
    masques_locaux = {}

    def traduire_selected(match):
        colonne, _, valeur = match.groups()
        if colonne not in dataframe.columns:
            raise KeyError(f"La variable relevant '{colonne}' est absente des tables jointes.")
        type_question = colonnes_survey.get(colonne, '')
        nom_liste = type_question.split()[-1] if type_question.startswith('select_one ') else None
        choix = next(
            (item for item in choix_par_liste.get(nom_liste, [])[:-1] if item['name'] == valeur),
            None
        )
        valeurs_attendues = {normaliser_reponse(valeur)}
        if choix:
            valeurs_attendues.update(choix['aliases'])
        nom_masque = f'relevant_{len(masques_locaux)}'
        masques_locaux[nom_masque] = dataframe[colonne].map(normaliser_reponse).isin(valeurs_attendues)
        return '@' + nom_masque
    texte = re.sub(
        r"selected\(\s*\$\{([^}]+)\}\s*,\s*(['\"])(.*?)\2\s*\)",
        traduire_selected, texte
    )
    texte = re.sub(r'\$\{([^}]+)\}', lambda m: '`' + m.group(1) + '`', texte)
    texte = re.sub(r'\band\b', '&', texte)
    texte = re.sub(r'\bor\b', '|', texte)
    resultat = dataframe.eval(texte, engine='python', local_dict=masques_locaux)
    if not isinstance(resultat, pd.Series):
        return pd.Series(bool(resultat), index=dataframe.index)
    return resultat.reindex(dataframe.index).fillna(False).astype(bool)


def enlever_condition_consentement(expression):
    if pd.isna(expression) or not str(expression).strip():
        return ''
    texte = re.sub(
        r"\(?\s*selected\(\s*\$\{volont\}\s*,\s*(['\"])oui\1\s*\)\s*\)?", '', str(expression), flags=re.IGNORECASE
    )
    texte = re.sub(r'^\s*&\s*|\s*&\s*$', '', texte)
    return re.sub(r'\(\s*\)', '', texte).strip()


def decrire_condition(expression, df_survey_clean, choix_par_liste):
    libelles = {
        str(row['name']): str(row.get('label', row['name'])).replace('\n', ' ').strip()
        for _, row in df_survey_clean.iterrows() if pd.notna(row.get('name'))
    }
    types = {
        str(row['name']): str(row.get('type', ''))
        for _, row in df_survey_clean.iterrows() if pd.notna(row.get('name'))
    }

    def traduire_selected(match):
        nom, valeur = match.groups()
        type_question = types.get(nom, '')
        nom_liste = type_question.split()[-1] if type_question.startswith('select_one ') else None
        choix = next((c for c in choix_par_liste.get(nom_liste, [])[:-1] if c['name'] == valeur), None)
        return f"{libelles.get(nom, nom)} = {choix['label'] if choix else valeur}"

    texte = re.sub(
        r"selected\(\s*\$\{([^}]+)\}\s*,\s*['\"]([^'\"]+)['\"]\s*\)",
        traduire_selected, expression
    )
    texte = re.sub(r'\$\{([^}]+)\}', lambda m: libelles.get(m.group(1), m.group(1)), texte)
    return texte.strip().strip('() ')


def evaluer_condition_ajoutee(nom_variable, dataframe):
    regle = recommandations_conditions['conditions'].get(nom_variable)
    if not regle:
        return None, None
    type_regle = regle['type']
    if type_regle == 'reponse_egale':
        variable = regle['variable']
        if variable not in dataframe.columns:
            return None, None
        masque = dataframe[variable].map(normaliser_reponse).eq(normaliser_reponse(regle['valeur']))
        return masque, regle['condition']
    if type_regle == 'variable_superieure_a_zero':
        variable = regle['variable']
        if variable not in dataframe.columns:
            return None, None
        masque = pd.to_numeric(dataframe[variable], errors='coerce').gt(0)
        return masque, regle['condition']
    if type_regle == 'effectif_menage_minimum':
        if regle['groupe_age'] == 'femmes_15_49':
            colonnes = ['f_15_17', 'f_18_49']
        else:
            tranches = ['0_5', '6_24', '25_59', '5_14', '15_17'] if regle['groupe_age'] == 'enfants' else ['18_49', '50_59', 'plus60']
            colonnes = [f'{sexe}_{tranche}' for sexe in ['h', 'f'] for tranche in tranches]
        if not all(col in dataframe.columns for col in colonnes):
            return None, None
        effectif = dataframe[colonnes].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=len(colonnes))
        return effectif.ge(regle['minimum']).fillna(False), regle['condition']
    if type_regle == 'validation_superieure':
        tranches = ['18_49', '50_59', 'plus60']
        colonnes = [f'{sexe}_{tranche}' for sexe in ['h', 'f'] for tranche in tranches]
        if nom_variable not in dataframe.columns or not all(col in dataframe.columns for col in colonnes):
            return None, None
        adultes = dataframe[colonnes].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=len(colonnes))
        valeur = pd.to_numeric(dataframe[nom_variable], errors='coerce')
        valide = valeur.isna() | adultes.isna() | valeur.le(adultes)
        return valide, regle['condition']
    raise ValueError(f"Type de condition recommandée non pris en charge : {type_regle}")


def afficher_composition_menage(df_cible, df_survey_clean):
    groupes = {
        'Hommes': [f'h_{tranche}' for tranche in ['0_5', '6_24', '25_59', '5_14', '15_17', '18_49', '50_59', 'plus60']],
        'Femmes': [f'f_{tranche}' for tranche in ['0_5', '6_24', '25_59', '5_14', '15_17', '18_49', '50_59', 'plus60']]
    }
    groupes = {sexe: [col for col in colonnes if col in df_cible.columns] for sexe, colonnes in groupes.items()}
    groupes = {sexe: colonnes for sexe, colonnes in groupes.items() if colonnes}
    colonnes_age = [col for colonnes in groupes.values() for col in colonnes]
    colonnes_total = [col for col in ['nb_personnes', 'nb_personne', 'nombre_personnes', 'nombre_personne', 'total_personnes', 'nb_personnes_total', 'nombre_personnes_total', 'taille_menage', 'personne'] if col in df_cible.columns]
    if not colonnes_total:
        for variantes in [['h', 'f'], ['hommes', 'femmes']]:
            if all(col in df_cible.columns for col in variantes):
                colonnes_total = variantes
                break

    print(f"\U0001f4ca Composition moyenne ({len(df_cible)} m\u00e9nages consentants) :")
    if colonnes_total:
        effectif_total = df_cible[colonnes_total].apply(pd.to_numeric, errors='coerce').sum(axis=1, min_count=len(colonnes_total))
        moyenne_total = effectif_total.mean()
        print(f'Nombre moyen total de personnes : {moyenne_total:.0f} (n={effectif_total.notna().sum()})')
        if moyenne_total > 20:
            afficher_alerte_coherence('La moyenne d\u00e9passe 20 personnes par m\u00e9nage. Le jeu de donn\u00e9es ne semble pas r\u00e9aliste ; les donn\u00e9es devront \u00eatre revues avec les agents MEAL afin de diagnostiquer une erreur technique, un manque de formation des agents et/ou un manque de compr\u00e9hension des enqu\u00eat\u00e9s.')
        return

    if not colonnes_age:
        return

    effectifs = df_cible[colonnes_age].apply(pd.to_numeric, errors='coerce')
    moyennes = effectifs.mean()
    moyennes_par_sexe = {sexe: moyennes[colonnes].sum() for sexe, colonnes in groupes.items()}
    enfants = [col for col in colonnes_age if re.fullmatch(r'[hf]_(0_5|6_24|25_59|5_14|15_17)', col)]
    personnes_agees = [col for col in colonnes_age if re.fullmatch(r'[hf]_plus60', col)]
    total_par_menage = effectifs.sum(axis=1, min_count=len(colonnes_age))
    moyenne_total = total_par_menage.mean()
    resume_composition = {'Nombre moyen total de personnes': moyenne_total}
    noms_moyenne_sexe = {'Femmes': 'de femmes', 'Hommes': 'd\u2019hommes'}
    for sexe in ['Femmes', 'Hommes']:
        if sexe in moyennes_par_sexe:
            resume_composition[f'Nombre moyen {noms_moyenne_sexe[sexe]}'] = moyennes_par_sexe[sexe]
    if enfants:
        resume_composition['Nombre moyen d\u2019enfants (moins de 18 ans)'] = moyennes[enfants].sum()
    if personnes_agees:
        resume_composition['Nombre moyen de personnes \u00e2g\u00e9es (60 ans et plus)'] = moyennes[personnes_agees].sum()
    print(pd.Series(resume_composition).round(0).astype('Int64').to_string())
    print(f'   - M\u00e9nages complets pour le total : {total_par_menage.notna().sum()} / {len(df_cible)}')

    if pd.notna(moyenne_total) and moyenne_total > 20:
            afficher_alerte_coherence('La moyenne d\u00e9passe 20 personnes par m\u00e9nage. Le jeu de donn\u00e9es ne semble pas r\u00e9aliste ; les donn\u00e9es devront \u00eatre revues avec les agents MEAL afin de diagnostiquer une erreur technique, un manque de formation des agents et/ou un manque de compr\u00e9hension des enqu\u00eat\u00e9s.')

    libelles = {
        str(row['name']): re.sub(r'\s+', ' ', str(row.get('label', row['name'])).replace('\n', ' ')).strip()
        for _, row in df_survey_clean.iterrows() if pd.notna(row.get('name'))
    }
    print('\U0001f4cb R\u00e9partition des tranches d\u2019\u00e2ge au sein de chaque sexe :')
    for sexe, colonnes in groupes.items():
        total_sexe = moyennes_par_sexe[sexe]
        lignes = [{
            'Tranche d\u2019\u00e2ge': libelles.get(col, col),
            'Personnes moyennes par m\u00e9nage': round(moyennes[col]),
            'Part du sexe (%)': round(moyennes[col] / total_sexe * 100, 2) if total_sexe else 0
        } for col in colonnes]
        print(f'\n{sexe}')
        print(pd.DataFrame(lignes).to_string(index=False))


def afficher_biens_ame(df_base, df_sec, df_survey_clean, choix_par_liste):
    lignes = []
    nombres_concernes = set()
    for _, row in df_survey_clean.iterrows():
        nom = row.get('name')
        if pd.isna(nom) or nom not in df_sec.columns:
            continue
        relevant = enlever_condition_consentement(row.get('relevant_combine'))
        eligibles = df_base
        if relevant:
            masque = evaluer_relevant(relevant, df_base, df_survey_clean, choix_par_liste)
            eligibles = df_base.loc[masque]
        nombres_concernes.add(len(eligibles))
        valeurs = pd.to_numeric(eligibles[nom], errors='coerce').dropna()
        label = re.sub(r'\s+', ' ', str(row.get('label', nom))).strip()
        if not label or label.lower() == 'nan':
            label = str(nom)
        statistiques_ame = recommandations_conditions.get('synthese_biens_ame', {}).get('statistiques', ['M\u00e9diane', 'Moyenne'])
        ligne = {'Bien': label}
        for statistique in statistiques_ame:
            valeur = valeurs.median() if normaliser_reponse(statistique) == 'mediane' else valeurs.mean()
            ligne[statistique] = '' if pd.isna(valeur) else f'{valeur:.0f}' if float(valeur).is_integer() else f'{valeur:.2f}'
        lignes.append(ligne)
    print('\n\U0001f4e6 Synth\u00e8se des biens AME : m\u00e9diane et moyenne par bien.')
    if len(nombres_concernes) == 1:
        print(f'Enqu\u00eat\u00e9s concern\u00e9s pour chaque bien : {next(iter(nombres_concernes))} / {len(df_base)}')
    else:
        print('Enqu\u00eat\u00e9s concern\u00e9s indiqu\u00e9s pour chaque bien, selon les conditions du XLSForm.')
    print(pd.DataFrame(lignes).to_string(index=False) if lignes else 'Aucun bien num\u00e9rique \u00e0 analyser.')


def executer_eda_toutes_tables_secondaires(db_path, tables_sec_liste, df_survey_clean, df_choices):
    choix_par_liste = preparer_choix(df_choices)
    conn = sqlite3.connect(db_path)
    df_meta = pd.read_sql('SELECT * FROM META_ENQUETE', conn)
    tables_disponibles = set(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'])
    df_contexte = pd.DataFrame({'id_menage': df_meta['id_menage']})
    for table_contexte in ['MENAGE', 'COMPOSITION_MENAGE']:
        if table_contexte in tables_disponibles:
            df_table_contexte = pd.read_sql(f'SELECT * FROM {table_contexte}', conn)
            df_contexte = df_contexte.merge(df_table_contexte, on='id_menage', how='left', suffixes=('', '_contexte'))

    for nom_table in tables_sec_liste:
        print(f"\n{'=' * 60}")
        print(f"\U0001f517 ANALYSE DE LA TABLE SECONDAIRE : {nom_table}")
        print('=' * 60)
        try:
            df_sec = pd.read_sql(f'SELECT * FROM {nom_table}', conn)
        except Exception as erreur:
            print(f'\u26a0 Impossible de charger la table {nom_table} : {erreur}')
            continue

        df_joint = pd.merge(df_sec, df_meta, on='id_menage', suffixes=('', '_meta'))
        colonnes_contexte = ['id_menage'] + [col for col in df_contexte.columns if col != 'id_menage' and col not in df_joint.columns]
        df_joint = df_joint.merge(df_contexte[colonnes_contexte], on='id_menage', how='left')
        if 'volont' in df_joint.columns:
            consentement = df_joint['volont'].astype(str).str.strip().str.lower().isin(['oui', 'yes', '1', 'true'])
            df_base = df_joint.loc[consentement].copy()
            print(f"\U0001f4cc Population de base (volont = oui) : {len(df_base)} / {len(df_joint)} enqu\u00eat\u00e9s")
        else:
            df_base = df_joint.copy()
            print(f'\U0001f4cc Total des enregistrements li\u00e9s : {len(df_base)}')
        if df_base.empty:
            print('\u26a0 Aucun enregistrement dans la population analys\u00e9e.')
            continue

        if nom_table == 'COMPOSITION_MENAGE':
            afficher_composition_menage(df_base, df_survey_clean)

        if nom_table == recommandations_conditions.get('synthese_biens_ame', {}).get('table'):
            afficher_biens_ame(df_base, df_sec, df_survey_clean, choix_par_liste)
            continue
        for _, row in df_survey_clean.iterrows():
            col_name = row.get('name')
            if pd.isna(col_name) or col_name not in df_sec.columns:
                continue
            col_name = str(col_name)
            if any(kw in col_name.lower() for kw in COLONNES_EXCLUES_TECHNIQUES + MOTS_CLES_EXCLUS_TEMPORELS):
                continue
            col_type = str(row.get('type', '')).lower()
            if 'date' in col_type or 'time' in col_type:
                continue
            if nom_table == 'COMPOSITION_MENAGE' and re.fullmatch(r'[hf]_(0_5|6_24|25_59|5_14|15_17|18_49|50_59|plus60)', col_name):
                continue

            label = row.get('label')
            label = re.sub(r'\s+', ' ', str(label)).strip() if pd.notna(label) else col_name
            if not label or label.lower() == 'nan':
                label = col_name
            print(f"\n{'-' * 50}")
            print(f"\U0001f50d Analyse : {label} (Type XLSForm : {col_type})")
            print('-' * 50)

            df_cible = df_base
            relevant_rule = enlever_condition_consentement(row.get('relevant_combine'))
            conditions = []
            descriptions_conditions = []
            conditions_ajoutees = []
            if relevant_rule:
                try:
                    masque = evaluer_relevant(relevant_rule, df_cible, df_survey_clean, choix_par_liste)
                    if masque is not None:
                        conditions.append(masque)
                        descriptions_conditions.append(decrire_condition(relevant_rule, df_survey_clean, choix_par_liste))
                except Exception as erreur:
                    print(f"\u26a0 Relevant non \u00e9valu\u00e9 ({relevant_rule}) : {erreur}. Analyse ignor\u00e9e.")
                    continue

            masque_technique, description_technique = evaluer_condition_ajoutee(col_name, df_cible)
            regle_technique = recommandations_conditions['conditions'].get(col_name)
            validation_incoherente = None
            if masque_technique is not None and regle_technique['type'] == 'validation_superieure':
                validation_incoherente = ~masque_technique
            elif masque_technique is not None:
                conditions.append(masque_technique)
            if description_technique:
                descriptions_conditions.append(description_technique)
                if masque_technique is not None and regle_technique['type'] != 'validation_superieure':
                    conditions_ajoutees.append(description_technique)

            regle_conditionnement = recommandations_conditions.get('alertes_conditionnement', {}).get(col_name)
            if regle_conditionnement:
                variable_parent = regle_conditionnement['variable_condition']
                lignes_parent = df_survey_clean.loc[
                    df_survey_clean['name'].astype(str).eq(variable_parent)
                ]
                if not lignes_parent.empty:
                    ligne_parent = lignes_parent.iloc[0]
                    relevant_parent = enlever_condition_consentement(
                        ligne_parent.get('relevant_combine')
                    )
                    if relevant_parent:
                        masque_parent = evaluer_relevant(
                            relevant_parent, df_base, df_survey_clean, choix_par_liste
                        )
                        conditions.append(masque_parent)
                        descriptions_conditions.append(
                            decrire_condition(relevant_parent, df_survey_clean, choix_par_liste)
                        )
                    regle_parent = recommandations_conditions['conditions'].get(variable_parent)
                    if regle_parent and regle_parent['type'] != 'validation_superieure':
                        masque_parent, description_parent = evaluer_condition_ajoutee(
                            variable_parent, df_base
                        )
                        if masque_parent is not None:
                            conditions.append(masque_parent)
                            descriptions_conditions.append(description_parent)
                            conditions_ajoutees.append(description_parent)

            for masque in conditions:
                df_cible = df_cible.loc[masque]
            total_eligible = len(df_cible)
            if regle_conditionnement:
                reponses_presentes_base = (
                    df_base[col_name].notna()
                    & df_base[col_name].astype(str).str.strip().ne('')
                )
                indices_eligibles = df_base.index.isin(df_cible.index)
                reponses_dans_condition = int(
                    (reponses_presentes_base & indices_eligibles).sum()
                )
                reponses_hors_condition = int(
                    (reponses_presentes_base & ~indices_eligibles).sum()
                )
                reponses_manquantes_condition = total_eligible - reponses_dans_condition
                if reponses_hors_condition or reponses_manquantes_condition:
                    afficher_alerte_coherence(
                        f'{col_name} : {total_eligible} enqu\u00eat\u00e9s sont concern\u00e9s apr\u00e8s application des conditions de la question parente et de la r\u00e9ponse \u00ab {regle_conditionnement["valeur_attendue"]} \u00bb \u00e0 {regle_conditionnement["variable_condition"]}; {reponses_dans_condition} r\u00e9ponses sont renseign\u00e9es parmi eux, {reponses_hors_condition} hors condition et {reponses_manquantes_condition} non renseign\u00e9es parmi les concern\u00e9s. V\u00e9rifier le relevant dans le formulaire.'
                    )
            non_concernes = len(df_base) - total_eligible
            for masque in conditions:
                df_cible = df_cible.loc[masque]
            total_eligible = len(df_cible)
            non_concernes = len(df_base) - len(df_base.loc[pd.concat(conditions, axis=1).all(axis=1)]) if conditions else 0
            condition_lisible = ' ; '.join(dict.fromkeys(filter(None, descriptions_conditions))) or 'Aucune condition suppl\u00e9mentaire'
            pct = total_eligible / len(df_base) * 100 if len(df_base) else 0
            print(f'Conditions attendues : {condition_lisible}')
            pct_affiche = f'{pct:.0f}%' if pct == 100 else f'{pct:.2f}%'
            print(f'   - Enqu\u00eat\u00e9s concern\u00e9s : {total_eligible} / {len(df_base)} ({pct_affiche})')
            print(f'   - Non concern\u00e9s (hors condition) : {non_concernes}')
            for condition_ajoutee in conditions_ajoutees:
                print(f"{recommandations_conditions['label_condition_ajoutee']} : {condition_ajoutee}")
            if validation_incoherente is not None:
                nombre_incoherents = int(validation_incoherente.loc[df_base.index].fillna(False).sum())
                print(f"{recommandations_conditions['label_condition_ajoutee']} : {description_technique}")
                print(f'   - Valeurs incoh\u00e9rentes (adultes en incapacit\u00e9 > adultes du m\u00e9nage) : {nombre_incoherents}')
            if total_eligible == 0:
                print('\u26a0 Aucun enqu\u00eat\u00e9 concern\u00e9 par cette condition.')
                continue

            serie = df_cible[col_name]
            masque_reponses_renseignees = serie.notna() & serie.astype(str).str.strip().ne('')
            non_renseignees = int(total_eligible - masque_reponses_renseignees.sum())
            print(f'R\u00e9ponses renseign\u00e9es : {int(masque_reponses_renseignees.sum())} ; non renseign\u00e9es : {non_renseignees}.')
            type_analyse = col_type
            nom_liste = None
            match_liste = re.match(r'^select_(?:one|multiple)\s+(\S+)', col_type)
            if match_liste:
                nom_liste = match_liste.group(1)
            elif col_type == 'text':
                type_inferre, nom_liste = inferer_type_texte(serie, choix_par_liste)
                if type_inferre:
                    type_analyse = type_inferre
                    print(f"\U0001f4a1 Inf\u00e9rence par l\u2019onglet choices : type {type_analyse}, name {nom_liste}.")

            est_choix_multiple = type_analyse.startswith('select_multiple')
            est_choix_unique = type_analyse.startswith('select_one')
            valeur_numerique = pd.to_numeric(serie, errors='coerce')
            valeurs_renseignees = serie.notna() & serie.astype(str).str.strip().ne('')
            frequences_numeriques = col_name in recommandations_conditions['repartition_frequences']
            est_numerique = (
                'integer' in col_type or 'decimal' in col_type or
                (not est_choix_unique and not est_choix_multiple and valeurs_renseignees.any()
                 and valeur_numerique[valeurs_renseignees].notna().all())
            )

            if est_numerique and frequences_numeriques:
                reponses_valides = valeur_numerique[valeur_numerique.notna()]
                if col_name in recommandations_conditions.get('alertes_oui_zero', {}):
                    nombre_zero = int(reponses_valides.eq(0).sum())
                    if nombre_zero:
                        parent = recommandations_conditions['alertes_oui_zero'][col_name]
                        afficher_alerte_coherence(
                            f'{nombre_zero} r\u00e9ponse(s) indiquent 0 fois alors que \u00ab {parent} \u00bb est oui. V\u00e9rifier une erreur technique, la coh\u00e9rence de la saisie par l\u2019enqu\u00eateur ou la compr\u00e9hension de la question par l\u2019enqu\u00eat\u00e9.'
                        )
                print('\U0001f4ca R\u00e9partition des r\u00e9ponses : les proportions sont calcul\u00e9es parmi les r\u00e9ponses renseign\u00e9es.')
                if not reponses_valides.empty:
                    frequences = reponses_valides.value_counts().sort_index()
                    frequences.index = [int(valeur) if float(valeur).is_integer() else valeur for valeur in frequences.index]
                    resume = pd.DataFrame({'Enqu\u00eat\u00e9s': frequences, 'Proportion des r\u00e9ponses (%)': (frequences / len(reponses_valides) * 100).round(2)})
                    print(resume.to_string())
                continue

            if est_numerique:
                valides = valeur_numerique.dropna()
                print(f'\U0001f4ca Statistiques num\u00e9riques (r\u00e9ponses valides : {len(valides)}, non renseign\u00e9es : {total_eligible - len(valides)}) :')
                if not valides.empty:
                    seuil = recommandations_conditions.get('alertes_moyenne', {}).get(col_name)
                    if seuil and valides.mean() > seuil['seuil_maximum']:
                        afficher_alerte_coherence(f"Moyenne observ\u00e9e : {valides.mean():.2f}. {seuil['message']}")
                    if col_name in recommandations_conditions.get('statistiques_moyenne_seule', []):
                        moyenne = valides.mean()
                        if col_name in recommandations_conditions.get('moyennes_entieres', []):
                            print(f'Moyenne : {moyenne:.0f}')
                        else:
                            print(f'Moyenne : {moyenne:.2f}')
                        continue
                    stats = valides.describe()[['mean', 'std', 'min', '50%', 'max']]
                    stats.index = ['Moyenne', '\u00c9cart-type', 'Minimum', 'M\u00e9diane', 'Maximum']
                    print(stats.round(2).to_string())
                else:
                    print('\u26a0 Aucune r\u00e9ponse num\u00e9rique valide.')
                continue

            choix_liste = choix_par_liste.get(nom_liste, []) if nom_liste else []
            if est_choix_multiple and choix_liste:
                selections = serie.apply(lambda valeur: extraire_choix_multiples(valeur, choix_liste) if pd.notna(valeur) else set())
                lignes = []
                for choix in choix_liste[:-1]:
                    effectif = sum(choix['name'] in selection for selection in selections)
                    lignes.append({'R\u00e9ponse': choix['label'], 'Enqu\u00eat\u00e9s': effectif, 'Pourcentage des concern\u00e9s (%)': round(effectif / total_eligible * 100, 2)})
                repondants = int(selections.map(bool).sum())
                print(f'\U0001f4cb Choix multiples : enqu\u00eat\u00e9s ayant s\u00e9lectionn\u00e9 au moins une r\u00e9ponse : {repondants} / {total_eligible}')
                print(pd.DataFrame(lignes).to_string(index=False))
                continue
            if est_choix_multiple and not choix_liste:
                print(f"\u26a0 Liste de choix '{nom_liste}' introuvable ; les r\u00e9ponses ne peuvent pas \u00eatre d\u00e9sagr\u00e9g\u00e9es.")

            if est_choix_unique:
                correspondance = {alias: choix['label'] for choix in choix_liste[:-1] for alias in choix['aliases']}
                serie_affichee = serie.map(lambda valeur: None if pd.isna(valeur) or not str(valeur).strip() else correspondance.get(normaliser_reponse(valeur), str(valeur)))
                reponses_renseignees = serie_affichee.dropna()
                frequences = reponses_renseignees.value_counts()
                resume = pd.DataFrame({'Effectif': frequences, 'Pourcentage (%)': (frequences / len(reponses_renseignees) * 100).round(2) if len(reponses_renseignees) else 0})
                print('\U0001f4cb Tableau de fr\u00e9quences :')
                print(resume.to_string())
                continue

            if col_type == 'text':
                series_text = serie.dropna().astype(str)
                series_text = series_text[series_text.str.strip() != '']
                print(f'\U0001f4dd Analyse texte libre ({len(series_text)} r\u00e9ponses) :')
                if not series_text.empty:
                    print(f"   - Longueur moyenne : {series_text.apply(lambda x: len(x.split())).mean():.1f} mots")
                    for commentaire, effectif in series_text.value_counts().head(5).items():
                        print(f'     * ({effectif}x) "{commentaire}"')
                else:
                    print('   - Aucune r\u00e9ponse textuelle enregistr\u00e9e.')

    conn.close()


# ==========================================
# 3. LANCEMENT GLOBAL
# ==========================================
df_choices = pd.read_excel(chemin_xlsform, sheet_name='choices')
executer_eda_toutes_tables_secondaires(db_path, tables_secondaires, df_survey_clean, df_choices)



🔗 ANALYSE DE LA TABLE SECONDAIRE : STATUT_MENAGE
📌 Population de base (volont = oui) : 46635 / 46635 enquêtés

--------------------------------------------------
🔍 Analyse : Quel est le type d'hebergement ? (Type XLSForm : select_one lieu_hab)
--------------------------------------------------
Conditions attendues : Aucune condition supplémentaire
   - Enquêtés concernés : 46635 / 46635 (100%)
   - Non concernés (hors condition) : 0
Réponses renseignées : 44323 ; non renseignées : 2312.
📋 Tableau de fréquences :
                            Effectif  Pourcentage (%)
herber                                               
Propre parcelle                 5654            12.76
Occupation gratuite             5608            12.65
Centre collectif                5574            12.58
Famille d'accueil               5546            12.51
Colocataire / Maison Louée      5543            12.51
Site spontanné                  5481            12.37
Locataire                       5459            1